In [ ]:
# %pip install "transformers>=4.37,<5" accelerate gradio

In [1]:
!nvidia-smi

Fri Nov 20 07:09:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import gradio as gr

gr.close_all()

In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

Device: cuda
GPU: Tesla T4
VRAM: 14.56317138671875 GB


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

# Set evaluation mode
model.eval()

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device: cuda
GPU: Tesla T4
Model loaded successfully!


In [5]:
print(model)

print("Total Parameters:", sum(p.numel() for p in model.parameters()))

print("Vocabulary Size:", model.config.vocab_size)
print("Hidden Size:", model.config.hidden_size)
print("Number of Layers:", model.config.num_hidden_layers)
print("Attention Heads:", model.config.num_attention_heads)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [6]:
import time
import torch

messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Give concise answers."},
    {"role": "user", "content": "What is artificial intelligence?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(device)

start_time = time.time()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

elapsed_time = time.time() - start_time

print(response)

print("\n========== PERFORMANCE ==========")
print(f"Inference Time: {elapsed_time:.2f} seconds")
print(f"Generated Tokens: {len(generated_tokens)}")
print(f"Tokens/sec: {len(generated_tokens)/elapsed_time:.2f}")

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves developing algorithms, models, and systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, perception, and language understanding. AI encompasses various fields including computer science, engineering, robotics, natural language processing, machine learning, and more.

========== PERFORMANCE ==========
Inference Time: 3.18 seconds
Generated Tokens: 83
Tokens/sec: 26.11


In [8]:
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [9]:
response = generate_response("What is artificial intelligence?")

print(response)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves creating software or hardware systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and natural language processing.

There are two main types of AI: weak AI, which is considered to be less intelligent than humans but capable of performing complex tasks; and strong AI, which is considered to be completely self-aware and capable of producing its own speech and thoughts.

Some examples of AI include:
- Self-driving cars
- Voice recognition technology
- Image and video recognition
- Chatbots and virtual assistants
- Natural language processing
- Robotics

Overall, AI has the potential to revolutionize many industries by enabling computers to learn from data, understand human languages, and make decisions based on context and information. However, it also raises ethical c

In [10]:
import gradio as gr

print(gr.__version__)
print(gr.__file__)
print(hasattr(gr, "ChatInterface"))

6.31.0
/usr/local/lib/python3.13/dist-packages/gradio/__init__.py
True


In [11]:
import os
import tempfile
from pathlib import Path

temp_dir = str(Path.home() / "gradio_temp")

os.makedirs(temp_dir, exist_ok=True)

os.environ["GRADIO_TEMP_DIR"] = temp_dir
os.environ["TMPDIR"] = temp_dir
os.environ["TEMP"] = temp_dir
os.environ["TMP"] = temp_dir

tempfile.tempdir = temp_dir

import gradio as gr
import gradio.routes

# Override the internal directory causing the error
gradio.routes.DEFAULT_TEMP_DIR = temp_dir

print("Gradio Version:", gr.__version__)
print("Temporary Directory:", gradio.routes.DEFAULT_TEMP_DIR)
print("Writable:", os.access(temp_dir, os.W_OK))

Gradio Version: 6.31.0
Temporary Directory: /root/gradio_temp
Writable: True


In [12]:
def chat(
    message,
    history,
    system_prompt,
    temperature,
    top_p,
    max_new_tokens
):

    # Convert structured content into plain text
    def extract_text(content):

        if isinstance(content, str):
            return content

        if isinstance(content, list):
            return "\n".join(
                item.get("text", "")
                for item in content
                if isinstance(item, dict)
                and item.get("type") == "text"
            )

        return str(content)

    # Initialize conversation
    messages = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    # Normalize conversation history
    for item in history:

        if item["role"] in ["user", "assistant"]:

            messages.append({
                "role": item["role"],
                "content": extract_text(item["content"])
            })

    # Add current user message
    messages.append({
        "role": "user",
        "content": extract_text(message)
    })

    # Apply Qwen chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenization
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    # Model inference
    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature),
            top_p=float(top_p),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract generated tokens
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    # Decode response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [16]:
import gradio as gr

gr.close_all()

system_prompt = "You are a helpful and intelligent AI assistant."

demo = gr.ChatInterface(

    fn=chat,

    title="My AI Assistant",

    description="""
    A conversational AI assistant powered by
    Hugging Face, Qwen, PyTorch and NVIDIA GPU.
    """,

    chatbot=gr.Chatbot(
        height=500
    ),

    textbox=gr.Textbox(
        placeholder="Ask me anything...",
        container=False
    ),

    additional_inputs=[

        gr.Textbox(
            value=system_prompt,
            label="System Prompt",
            lines=3
        ),

        gr.Slider(
            minimum=0.1,
            maximum=2.0,
            value=0.7,
            step=0.1,
            label="Temperature"
        ),

        gr.Slider(
            minimum=0.1,
            maximum=1.0,
            value=0.9,
            step=0.05,
            label="Top P"
        ),

        gr.Slider(
            minimum=64,
            maximum=2048,
            value=512,
            step=64,
            label="Maximum New Tokens"
        )

    ],

    examples=[
        ["What is artificial intelligence?", system_prompt, 0.7, 0.9, 512],
        ["Explain deep learning with examples.", system_prompt, 0.7, 0.9, 512],
        ["Write a Python program for linear regression.", system_prompt, 0.7, 0.9, 512],
        ["What is the difference between CNN and RNN?", system_prompt, 0.7, 0.9, 512],
        ["Explain the Transformer architecture.", system_prompt, 0.7, 0.9, 512]
    ],

    cache_examples=False,
    analytics_enabled=False
)

demo.launch(
    server_name="127.0.0.1",
    share=True,
    inline=False
)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ee98b365b9af571fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
